In [ ]:
# Lab type: debug
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Permission-Aware and Multi-Tenant Retrieval
# Task: The multi-tenant retrieval service below contains 3 bugs — one
# recall bug, one leak, and one authorisation-staleness bug. Find and fix
# each one; the demo queries make each visible if you look.

# Lab: Debugging a Multi-Tenant Retriever

Two tenants share an index: **acme** (2 documents) and **globex** (10 documents). Tenant size matters — one of the bugs only hurts the small tenant.

**Outputs are cleared.** Run every cell top to bottom.

## Setup: a two-tenant corpus

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

In [ ]:
# acme's private docs (2) and globex's private docs (10, reusing the KB)
TENANT_DOCS = [
    ("acme-contract", "acme",
     "Contracts > Acme: Acme Corp's enterprise contract renews on March 1 "
     "with a 15 percent negotiated discount and a 99.9 percent SLA."),
    ("acme-tickets", "acme",
     "Support > Acme: Acme reported an SSO outage last quarter, resolved "
     "by rotating the SAML certificate."),
] + [(d, "globex", f"{h}: {t}") for d, h, t in CORPUS[:10]]

T_IDS = [d for d, _, _ in TENANT_DOCS]
T_TENANT = {d: ten for d, ten, _ in TENANT_DOCS}
T_TEXTS = [t for _, _, t in TENANT_DOCS]
T_EMB = embed(T_TEXTS)
print(f"{len(TENANT_DOCS)} docs: "
      f"{sum(1 for d in T_IDS if T_TENANT[d]=='acme')} acme, "
      f"{sum(1 for d in T_IDS if T_TENANT[d]=='globex')} globex")

## The service (contains 3 bugs)

In [ ]:
# --- AI-GENERATED MULTI-TENANT RETRIEVAL SERVICE ---
# Review this code — is it correct?
query_cache = {}

def search_for_tenant(query, tenant, k=3):
    # cache retrievals to save embedding calls
    if query in query_cache:                     # (bug candidate)
        return query_cache[query]
    scores = T_EMB @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]         # (bug candidate)
    results = [T_IDS[i] for i in order]
    allowed = [d for d in results
               if T_TENANT[d] == tenant]          # (bug candidate)
    query_cache[query] = allowed
    return allowed

print("acme asks about its own contract:")
print("  ", search_for_tenant("when does our contract renew", "acme"))
print("globex asks the same question:")
print("  ", search_for_tenant("when does our contract renew", "globex"))
print("acme asks a generic product question:")
print("  ", search_for_tenant("how long is event data kept", "acme"))

## Bug 1: the second print

globex asked about *its* contract and got acme's contract doc (or acme's cached results). Which line leaks across tenants, and why is this worse than a wrong answer?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**The bug:** `query_cache` is keyed on the query alone — no tenant in the key. globex's identical query returns acme's cached result list.

**Why it causes wrong behaviour:** this is a cross-tenant data leak, not a relevance bug: acme's private doc IDs (and whatever is fetched with them downstream) are served to globex. It is also exactly the class of defect post-filter architectures invite — state between search and filter that doesn't carry the authorisation context.

**Correct approach:** key the cache on `(tenant, query)` — or drop the cache entirely until the filter placement is fixed (Bug 3).

</details>

## Bug 2: the third print

acme asked a generic product question. The corpus has a perfectly good answer (`data-retention`) — why did acme get nothing (or nearly nothing), and which tenant will *never* notice this bug?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**The bug:** the search takes the global top-k *first* (`np.argsort(scores)[::-1][:k]` over all tenants' docs) and filters afterwards.

**Why it causes wrong behaviour:** the candidate budget is spent on chunks acme cannot see. Wait — here the leak-filter keeps only acme docs out of a top-3 dominated by globex's larger corpus, so acme (2 of 12 docs) gets starved. globex, owning most of the corpus, never notices — which is why post-filtering looks like a "small tenant relevance problem" in production and gets mis-fixed by raising k.

**Correct approach:** pre-filter — restrict the score computation (or the index itself, via per-tenant namespaces) to the tenant's own documents *before* taking top-k.

</details>

## Bug 3: what nobody asked yet

Suppose acme's contractor loses access, or a doc moves from `globex` to a restricted tenant. Where does this service's authorisation state live, and what is wrong with that?

**Explain the bug:**

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**The bug:** `T_TENANT` was captured once at ingestion and — worse — results already sit in `query_cache` beyond any permission check at all: a revocation changes nothing already cached.

**Why it causes wrong behaviour:** permission staleness is measured in exposure, not relevance. Cached result lists and ingestion-time ACL snapshots keep serving access that has been revoked.

**Correct approach:** fast-path permission changes through the index update pipeline (ahead of content edits), keep volatile ACLs out of long-lived caches, and resolve fine-grained permissions against the live authorisation service at query time (as defence in depth on top of the tenant pre-filter).

</details>

## The fixed service + a leak regression test

In [ ]:
# Fix for Bugs 1-3: tenant-scoped pre-filter, tenant-keyed (minimal)
# cache, and a cross-tenant leak test to gate regressions.
def search_for_tenant_fixed(query, tenant, k=3):
    mask = np.array([T_TENANT[d] == tenant for d in T_IDS])
    idx = np.where(mask)[0]                    # tenant's docs only
    scores = T_EMB[idx] @ embed([query])[0]    # pre-filtered search
    order = idx[np.argsort(scores)[::-1][:k]]
    return [T_IDS[i] for i in order]

print("acme, generic question, full recall now:")
print("  ", search_for_tenant_fixed("how long is event data kept", "acme"))
print("globex, contract question, no acme docs:")
print("  ", search_for_tenant_fixed("when does our contract renew", "globex"))

def leak_test():
    probes = ["when does our contract renew", "sso outage",
              "negotiated discount", "acme SLA"]
    for q in probes:
        for d in search_for_tenant_fixed(q, "globex", k=5):
            assert T_TENANT[d] == "globex", f"LEAK: {d} via {q!r}"
    return "leak test passed: no acme doc reachable from globex"

print(leak_test())

## Summary

1. Post-filtering spends the candidate budget on chunks the tenant _______.
2. The leak was a cache keyed on _______ alone.
3. A leak is a retrieval result, so it is _______ like one — add the cross-tenant test to the regression gate.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **cannot see** — small tenants starve, large tenants never notice.
2. **the query** — no tenant in the key.
3. **testable**

</details>